In [275]:
import glob
import pandas as pd
import numpy as np

In [276]:
# Paths are relative to this notebook (notebooks/), so they work on any computer that clones the repo
path = "../data/raw/"
out_path = "../data/results/"

race_list=glob.glob(pathname="*_R.csv", root_dir=path)

race_csv_combined=[]
for r in race_list:
    race_csv_combined.append(pd.read_csv(path + r))
races_combined = pd.concat(race_csv_combined, ignore_index=True)
races_combined.to_csv(path_or_buf=out_path + 'results.csv', index=False)
races_combined.shape


(2278, 26)

In [277]:
# Qualifying
quali_list = glob.glob(pathname="*_Q.csv", root_dir=path)

quali_csv_combined = []
for q in quali_list:
    quali_csv_combined.append(pd.read_csv(path + q))
quali_combined = pd.concat(quali_csv_combined, ignore_index=True)

# Data fix: FastF1 has no Q1-Q3 times for Verstappen at 2025 Miami (round 6), where he took pole.
# Times entered by hand from the official results on formula1.com. Fixed here, not in data/raw/,
# so the raw files stay exactly as downloaded and the fix is reapplied on every run.
ver_miami = (quali_combined['Year'] == 2025) & (quali_combined['Round'] == 6) & (quali_combined['Abbreviation'] == 'VER')
assert ver_miami.sum() == 1, 'expected exactly one VER row for 2025 Miami'
quali_combined.loc[ver_miami, ['Q1', 'Q2', 'Q3']] = ['0 days 00:01:26.870000', '0 days 00:01:26.643000', '0 days 00:01:26.204000']

quali_combined.to_csv(path_or_buf=out_path + 'quali.csv', index=False)
quali_combined.shape


(2279, 26)

In [278]:
# Sprint races
sprint_list = glob.glob(pathname="*_S.csv", root_dir=path)

sprint_csv_combined = []
for s in sprint_list:
    sprint_csv_combined.append(pd.read_csv(path + s))
sprint_combined = pd.concat(sprint_csv_combined, ignore_index=True)
sprint_combined.to_csv(path_or_buf=out_path + 'sprint.csv', index=False)
sprint_combined.shape


(552, 26)

In [279]:
# Sprint qualifying: SQ (2024 on) and SS (2023's "Sprint Shootout") are the same session under two names
sprint_quali_list = glob.glob(pathname="*_S[QS].csv", root_dir=path)

sprint_quali_csv_combined = []
for sq in sprint_quali_list:
    sprint_quali_csv_combined.append(pd.read_csv(path + sq))
sprint_quali_combined = pd.concat(sprint_quali_csv_combined, ignore_index=True)
sprint_quali_combined.to_csv(path_or_buf=out_path + 'sprint_quali.csv', index=False)
sprint_quali_combined.shape


(492, 26)

In [280]:
# Free practice: FP1, FP2 and FP3 together (the Session column tells them apart)
practice_list = glob.glob(pathname="*_FP?.csv", root_dir=path)

practice_csv_combined = []
for p in practice_list:
    practice_csv_combined.append(pd.read_csv(path + p))
practice_combined = pd.concat(practice_csv_combined, ignore_index=True)
practice_combined.to_csv(path_or_buf=out_path + 'practice.csv', index=False)
practice_combined.shape


(5795, 26)

In [281]:
qualis = pd.read_csv(out_path +  'quali.csv')
qualis[['Q1','Q2','Q3']] = qualis[['Q1','Q2','Q3']].apply(lambda c: pd.to_timedelta(c).dt.total_seconds())
cols=list(qualis.columns)
qualis.insert(loc = (qualis.columns.get_loc('Q3')+1), column = 'Best', value=np.nanmin(qualis[['Q1','Q2','Q3']].to_numpy(),axis=1)) 




C:\Users\Hridy\AppData\Local\Temp\ipykernel_17852\1667155139.py:4: RuntimeWarning: All-NaN slice encountered
  qualis.insert(loc = (qualis.columns.get_loc('Q3')+1), column = 'Best', value=np.nanmin(qualis[['Q1','Q2','Q3']].to_numpy(),axis=1))


In [282]:
#Podium Target and Classified Flag
races = pd.read_csv(out_path + 'results.csv')
races.insert(loc = races.columns.get_loc('Position')+1, column = 'Podium', value = np.where(races['Position'].to_numpy() <= 3, 1, 0))
races.columns.tolist()

exclude_from_classified = ['W','D','R','E','N']
races.insert(loc = races.columns.get_loc('ClassifiedPosition')+1, column = 'ClassifiedFlag', value = np.where(~races['ClassifiedPosition'].isin(exclude_from_classified), 1, 0))



races[['ClassifiedPosition', 'ClassifiedFlag']]

,ClassifiedPosition,ClassifiedFlag
0,1,1
1,2,1
2,3,1
3,4,1
4,5,1
...,...,...
2273,18,1
2274,19,1
2275,20,1
2276,R,0


In [283]:
#Implementing the grid_position transform
#Sets a pitlane start (GridPosition == 0) to the number of cars in that race (20, or 22 in 2026)
#Writes to a new 'Grid' column so the raw GridPosition stays untouched and re-running this cell is safe
#Withdrawn drivers (GridPosition NaN) stay NaN: they never started and get dropped later
field_size = races.groupby(['Year','Round']).transform('size')
races['Grid'] = np.where(races['GridPosition'] == 0, field_size, races['GridPosition'])

#Check: every pit-lane starter, before and after
races.loc[races['GridPosition'] == 0, ['Year','Round','Abbreviation','GridPosition','Grid']]
races[['Abbreviation','GridPosition','Grid','Year','Round']].iloc[70:77]


,Abbreviation,GridPosition,Grid,Year,Round
70,ALB,18.0,18.0,2022,4
71,GAS,17.0,17.0,2022,4
72,HAM,14.0,14.0,2022,4
73,OCO,16.0,16.0,2022,4
74,ZHO,0.0,20.0,2022,4
75,LAT,19.0,19.0,2022,4
76,MSC,10.0,10.0,2022,4


In [284]:
#Baseline: what you'd score by just trusting the starting grid
#Measure 1, pole_to_win: share of races won from Grid 1
#Measure 2, grid_top3_hits: average number of the grid top 3 who finished on the podium (0 to 3)
#Grid is the actual starting grid (after penalties and pit-lane starts), not raw qualifying order
def grid_baseline(season):
    # One True/False per driver row; summing booleans counts the Trues
    pole_won = (season['Grid'] == 1) & (season['Position'] == 1)
    top3_on_podium = (season['Grid'] <= 3) & (season['Podium'] == 1)

    # Collapse to one row per race: did pole win (0 or 1), how many top-3 starters made the podium
    per_race = pd.DataFrame({'Round': season['Round'],
                             'PoleWon': pole_won,
                             'Top3Podium': top3_on_podium}).groupby('Round').sum()
    return per_race

per_race_2025 = grid_baseline(races[races['Year'] == 2025])
pole_to_win_2025 = per_race_2025['PoleWon'].mean()
grid_top3_hits_2025 = per_race_2025['Top3Podium'].mean()

print(f"2025: {len(per_race_2025)} races, pole won {pole_to_win_2025:.1%}, "
      f"grid top 3 on podium {grid_top3_hits_2025:.2f} of 3 per race")
per_race_2025.T

2025: 24 races, pole won 66.7%, grid top 3 on podium 2.25 of 3 per race


Round,1,2,3,4,5,6,7,8,9,10,...,15,16,17,18,19,20,21,22,23,24
PoleWon,1,1,1,1,0,0,0,1,1,1,...,1,1,1,1,1,1,1,0,0,1
Top3Podium,2,3,3,2,2,1,2,3,2,2,...,2,3,2,2,3,2,2,1,2,3


In [285]:
#Same baseline on 2026 so far (the reality check for the new rules)
per_race_2026 = grid_baseline(races[races['Year'] == 2026])
pole_to_win_2026 = per_race_2026['PoleWon'].mean()
grid_top3_hits_2026 = per_race_2026['Top3Podium'].mean()

#The four README numbers in one table
baseline = pd.DataFrame({
    'Races': [len(per_race_2025), len(per_race_2026)],
    'PoleToWin': [pole_to_win_2025, pole_to_win_2026],
    'GridTop3OnPodium': [grid_top3_hits_2025, grid_top3_hits_2026],
}, index=[2025, 2026])
baseline

,Races,PoleToWin,GridTop3OnPodium
2025,24,0.666667,2.25
2026,20,0.700000,1.80


In [ ]:
#Save the cleaned tables so 03_features can start from them without re-running this notebook
qualis.to_csv(out_path + 'quali_clean.csv', index=False)
races.to_csv(out_path + 'races_clean.csv', index=False)